# RACE Project Runner (Colab + VSCode)

This notebook is the **single place** to run setup, preprocessing, Model A training, and quick checks.

If you are using Colab extension in VSCode, run cells top-to-bottom.

## What this notebook does
- Verifies dataset files exist in `data/raw/`
- Runs preprocessing (`src.preprocessing`)
- Trains Model A (`src.model_a_train`)
- Shows saved metrics and generated artifacts

## Expected dataset files
- `data/raw/train.csv`
- `data/raw/dev.csv` (or `data/raw/val.csv`)
- `data/raw/test.csv`

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
print('Current working directory:', ROOT)
print('Python:', sys.version)


def find_project_root(start: Path) -> Path | None:
    # 1) Walk up from current directory
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'requirements.txt').exists() and (p / 'src').exists():
            return p

    # 2) Common Colab clone locations
    candidates = [
        Path('/content/Intelligent-Reading-Comprehension-and-Quiz-Generation-System-using-Machine-Learning-Neural-Networks/race_rc_project'),
        Path('/content/race_rc_project'),
    ]
    for p in candidates:
        if (p / 'requirements.txt').exists() and (p / 'src').exists():
            return p

    # 3) Shallow search under /content for race_rc_project
    base = Path('/content')
    if base.exists():
        for p in base.rglob('race_rc_project'):
            if (p / 'requirements.txt').exists() and (p / 'src').exists():
                return p
    return None

project_root = find_project_root(ROOT)
if project_root is None:
    raise FileNotFoundError(
        'Could not find race_rc_project root. Clone the repo first, then rerun.\n'
        'Example:\n'
        '!git clone https://github.com/Hudiyahaha/Intelligent-Reading-Comprehension-and-Quiz-Generation-System-using-Machine-Learning-Neural-Networks.git\n'
        '%cd /content/Intelligent-Reading-Comprehension-and-Quiz-Generation-System-using-Machine-Learning-Neural-Networks/race_rc_project'
    )

if Path.cwd().resolve() != project_root.resolve():
    %cd {project_root}

print('Project root:', Path.cwd())

In [ ]:
# Install dependencies (safe to re-run)
!pip -q install -r requirements.txt

## Data setup options for Colab

Use **one** of the two options below before running preprocessing:

1. Upload files directly in Colab session (temporary)
2. Copy files from Google Drive (persistent)

In [ ]:
# Option 1 (Colab): Upload CSV files from your machine into this session
# This is temporary storage; files disappear when runtime resets.

from pathlib import Path
import shutil

raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files
    uploaded = files.upload()  # select train.csv, dev.csv(or val.csv), test.csv
    for name in uploaded.keys():
        src = Path(name)
        dst = raw_dir / src.name
        shutil.move(str(src), str(dst))
        print('Saved:', dst)
except Exception as e:
    print('Not running in Colab or upload failed:', e)
    print('You can skip this if you will use Option 2 (Drive).')

In [ ]:
# Option 2 (Colab): Copy CSV files from Google Drive into data/raw
# Update DRIVE_DATA_DIR to your folder containing train.csv/dev.csv(or val.csv)/test.csv

from pathlib import Path
import shutil

DRIVE_DATA_DIR = '/content/drive/MyDrive/race_data'  # change this path
raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    src_dir = Path(DRIVE_DATA_DIR)
    if not src_dir.exists():
        raise FileNotFoundError(f'Drive path not found: {src_dir}')
    for fname in ['train.csv', 'dev.csv', 'val.csv', 'test.csv']:
        src = src_dir / fname
        if src.exists():
            dst = raw_dir / fname
            shutil.copy2(src, dst)
            print('Copied:', src, '->', dst)
except Exception as e:
    print('Drive copy step skipped/failed:', e)
    print('Use Option 1 or fix DRIVE_DATA_DIR and rerun this cell.')

In [ ]:
# Check dataset files
from pathlib import Path

raw_dir = Path('data/raw')
expected = ['train.csv', 'test.csv']
optional_val = ['dev.csv', 'val.csv']

for f in expected:
    print(f, '->', (raw_dir / f).exists())
print('dev/val exists ->', any((raw_dir / f).exists() for f in optional_val))

if not (raw_dir / 'train.csv').exists() or not (raw_dir / 'test.csv').exists() or not any((raw_dir / f).exists() for f in optional_val):
    raise FileNotFoundError('Missing required CSVs in data/raw. Add train/test + dev or val first.')

In [ ]:
# Run preprocessing
!python -m src.preprocessing --raw-dir data/raw --processed-dir data/processed

In [ ]:
# Inspect preprocessing outputs
manifest_path = Path('data/processed/manifest.json')
if not manifest_path.exists():
    raise FileNotFoundError('manifest.json not found. Preprocessing may have failed.')

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(json.dumps({
    'mcq_splits': manifest.get('mcq_splits'),
    'verify_splits': manifest.get('splits'),
    'ohe_feature_dim': manifest.get('ohe_feature_dim'),
    'tfidf_feature_dim': manifest.get('tfidf_feature_dim')
}, indent=2))

In [ ]:
# Train Model A (full)
!python -m src.model_a_train --processed-dir data/processed --output-dir models/model_a/traditional

In [ ]:
# Faster debug run (optional). Uncomment to use.
# !python -m src.model_a_train --processed-dir data/processed --output-dir models/model_a/traditional_debug --max-train-rows 20000 --generation-max-train-mcq 5000

In [ ]:
# Read Model A metrics
metrics_path = Path('models/model_a/traditional/metrics_summary.json')
if not metrics_path.exists():
    raise FileNotFoundError('Model A metrics file not found. Train step may have failed.')

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print('Config:')
print(json.dumps(metrics['config'], indent=2))
print('\nAvailable sections:', list(metrics.keys()))
print('\nVerification models:', list(metrics['results'].keys()))
if 'generation' in metrics:
    print('\nGeneration summary:')
    print(json.dumps(metrics['generation'], indent=2))

In [ ]:
# Quick artifact check
artifact_dir = Path('models/model_a/traditional')
for p in sorted(artifact_dir.glob('*')):
    print(p.name)